In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!java --version

openjdk 11.0.28 2025-07-15
OpenJDK Runtime Environment (build 11.0.28+6-post-Ubuntu-1ubuntu122.04.1)
OpenJDK 64-Bit Server VM (build 11.0.28+6-post-Ubuntu-1ubuntu122.04.1, mixed mode, sharing)


In [ ]:
!pip list | grep pyspark

pyspark                               3.5.1


In [ ]:
# Remove old Java
!apt-get remove openjdk-* -y

# Install OpenJDK 17
!apt-get update -q
!apt-get install openjdk-17-jdk -y

# Set JAVA_HOME
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Verify
!java -version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'openjdk-11-jdk' for glob 'openjdk-*'
Note, selecting 'openjdk-11-jre' for glob 'openjdk-*'
Note, selecting 'openjdk-11-jre-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-19-jre-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-8-jre-zero' for glob 'openjdk-*'
Note, selecting 'openjdk-11-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-21-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-19-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-21-demo' for glob 'openjdk-*'
Note, selecting 'openjdk-18-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-17-dbg' for glob 'openjdk-*'
Note, selecting 'openjdk-17-doc' for glob 'openjdk-*'
Note, selecting 'openjdk-18-dbg' for glob 'openjdk-*'
Note, selecting 'openjdk-17-jdk' for glob 'openjdk-*'
Note, selecting 'openjdk-18-doc' for glob 'openjdk-*'
Note, selecting 'openjdk-17-jre' f

In [ ]:
!pip install --upgrade pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 14.0 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813800 sha256=142e3a5c9e91ddd265eb32c2fd22a9660887d49236eb62c2b718d5237bfba98b
  Stored in directory: /root/.cache/pip/wheels/31/9f/68/f89fb34ccd886909be7d0e390eaaf97f21efdf540c0ee8dbcd
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.1
    Uninstalling pyspark-3.5.1:
      Successfully uninstalled pyspark-3.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

## Sobre os dados

O arquivo CSV contém eventos 'click' ou 'view' no tempo, de usuários em anúncios de determinadas campanhas.

**Descrição das colunas:**  
timestamp,user_id,action,adId,campaignId

**Amostra:**  
2016-09-21 22:11:00,7c74953c-66cc-48bd-9d02-a02bf039cf3f,click,adId_09,campaignId_01  
2016-06-25 18:29:00,676a083e-2f8e-4ff2-9ec2-270f7f9d6033,view,adId_09,campaignId_02  
2016-02-14 19:03:00,77158997-0dfa-48b7-9149-973dc151ef8d,click,adId_02,campaignId_02  
2016-03-26 06:27:00,78aa2467-b502-413b-94e9-04ec8210bd13,click,adId_07,campaignId_03

**Nome do arquivo CSV:**  
data/ad_action.csv

## Sobre as questões

As questões devem ser respondidas usando alguma API da tecnologia Spark, exceto a API "Pandas API on Spark".

Quando utilizar uma action do Spark tenha cuidado para evitar estouro de memória, sempre imaginado que vai executar o código com uma grande massa de dados.

Mesmo que não consiga terminar alguma questão, favor enviar, porque parte do código pode valer alguma pontuação.

In [ ]:
import os
import pyspark.sql.functions as F
import pyspark.sql.types as T

from pyspark.sql import SparkSession

os.environ['PYSPARK_SUBMIT_ARGS'] = '\
    --driver-memory 2G \
    --executor-memory 2G \
    pyspark-shell'

In [ ]:
spark = SparkSession.builder\
    .master("local[*]")\
    .getOrCreate()
data_spark = spark.read.csv('drive/MyDrive/data/ad_action.csv', header=False, inferSchema=True)\
    .toDF('timestamp', 'user_id', 'action', 'adId', 'campaignId')
data_spark.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- action: string (nullable = true)
 |-- adId: string (nullable = true)
 |-- campaignId: string (nullable = true)



In [ ]:
data_spark.show(5)

+-------------------+--------------------+------+-------+-------------+
|          timestamp|             user_id|action|   adId|   campaignId|
+-------------------+--------------------+------+-------+-------------+
|2016-09-21 22:11:00|7c74953c-66cc-48b...| click|adId_09|campaignId_01|
|2016-06-25 18:29:00|676a083e-2f8e-4ff...|  view|adId_09|campaignId_02|
|2016-02-14 19:03:00|77158997-0dfa-48b...| click|adId_02|campaignId_02|
|2016-03-26 06:27:00|78aa2467-b502-413...| click|adId_07|campaignId_03|
|2016-01-02 04:57:00|fef9a98c-d73e-48e...|  view|adId_02|campaignId_02|
+-------------------+--------------------+------+-------+-------------+
only showing top 5 rows


In [ ]:
# Descomente para desligar clusters

# spark.stop()

## 1) Quais são as top 3 campanhas que geraram mais eventos? Ordene pela quantidade de eventos (2,5 pontos)

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

In [ ]:
data_spark.groupBy('campaignId')\
.count()\
.orderBy('count', ascending=False)\
.limit(3)\
.toPandas()

,campaignId,count
0,campaignId_02,91216
1,campaignId_03,87036
2,campaignId_01,76461


## 2) Qual campanha teve mais clicks? (2,5 pontos)

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

In [ ]:
data_spark.where(F.col('action')=='click')\
.groupBy('campaignId')\
.count()\
.orderBy('count', ascending=False)\
.take(1)[0]['campaignId']

'campaignId_02'

## 3) Qual mês teve o maior total de eventos acumulado? (2,5 pontos)

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

In [ ]:
df_month = data_spark.withColumn('month', F.month('timestamp'))
df_month.show(5)

+-------------------+--------------------+------+-------+-------------+-----+
|          timestamp|             user_id|action|   adId|   campaignId|month|
+-------------------+--------------------+------+-------+-------------+-----+
|2016-09-21 22:11:00|7c74953c-66cc-48b...| click|adId_09|campaignId_01|    9|
|2016-06-25 18:29:00|676a083e-2f8e-4ff...|  view|adId_09|campaignId_02|    6|
|2016-02-14 19:03:00|77158997-0dfa-48b...| click|adId_02|campaignId_02|    2|
|2016-03-26 06:27:00|78aa2467-b502-413...| click|adId_07|campaignId_03|    3|
|2016-01-02 04:57:00|fef9a98c-d73e-48e...|  view|adId_02|campaignId_02|    1|
+-------------------+--------------------+------+-------+-------------+-----+
only showing top 5 rows


In [ ]:
df_month.groupBy('month')\
.count()\
.orderBy('count', ascending=False)\
.limit(1)\
.toPandas()

,month,count
0,1,25800


## 4) Nas situações onde existe um evento de view seguido de um evento de click criados pelo mesmo usuário no mesmo anúncio e campanha, quais são os 5 pares de anúncio e campanha com menores médias de tempo entre os dois eventos (2,5 pontos)

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

In [ ]:
views = data_spark.where(F.col('action')=='view').withColumnRenamed('timestamp', 'view_time')\
.orderBy('view_time')
views.show(5)

+-------------------+--------------------+------+-------+-------------+
|          view_time|             user_id|action|   adId|   campaignId|
+-------------------+--------------------+------+-------+-------------+
|2016-01-01 00:07:00|4fde5dfa-ccde-4c2...|  view|adId_07|campaignId_02|
|2016-01-01 00:15:00|218dc6e2-1621-4b4...|  view|adId_04|campaignId_01|
|2016-01-01 00:20:00|1994f3e6-591b-42a...|  view|adId_04|campaignId_02|
|2016-01-01 00:27:00|9b3b575a-4c7a-4a1...|  view|adId_01|campaignId_01|
|2016-01-01 00:28:00|ae224ccd-b186-487...|  view|adId_07|campaignId_03|
+-------------------+--------------------+------+-------+-------------+
only showing top 5 rows


In [ ]:
clicks = data_spark.where(F.col('action')=='click').withColumnRenamed('timestamp', 'click_time')\
.orderBy('click_time')
clicks.show(5)

+-------------------+--------------------+------+-------+-------------+
|         click_time|             user_id|action|   adId|   campaignId|
+-------------------+--------------------+------+-------+-------------+
|2016-01-01 00:00:00|c577e717-1be3-4e6...| click|adId_06|campaignId_02|
|2016-01-01 00:07:00|0fb092f2-6721-4dd...| click|adId_10|campaignId_03|
|2016-01-01 00:11:00|2dbd9392-768e-4ba...| click|adId_03|campaignId_03|
|2016-01-01 00:12:00|cd3e0222-0796-450...| click|adId_09|campaignId_01|
|2016-01-01 00:13:00|909401e9-55ef-4dd...| click|adId_02|campaignId_01|
+-------------------+--------------------+------+-------+-------------+
only showing top 5 rows


In [ ]:
from pyspark.sql import Window

In [ ]:
#tem todas as possibilidades de view_time -> click_time para o mesmo user_id, adId, campaignId
joined = views.join(
    clicks,
    on=["user_id", "adId", "campaignId"],
    how="left"
)
joined.show(5)

+--------------------+-------+-------------+-------------------+------+-------------------+------+
|             user_id|   adId|   campaignId|          view_time|action|         click_time|action|
+--------------------+-------+-------------+-------------------+------+-------------------+------+
|39ba8e01-f1bb-491...|adId_02|campaignId_03|2016-01-01 09:26:00|  view|2016-08-03 11:29:00| click|
|39ba8e01-f1bb-491...|adId_02|campaignId_03|2016-01-01 09:26:00|  view|2016-12-28 10:35:00| click|
|39ba8e01-f1bb-491...|adId_02|campaignId_03|2016-01-01 09:26:00|  view|2016-12-04 21:53:00| click|
|39ba8e01-f1bb-491...|adId_02|campaignId_03|2016-01-01 09:26:00|  view|2016-04-24 20:26:00| click|
|39ba8e01-f1bb-491...|adId_02|campaignId_03|2016-01-01 09:26:00|  view|2016-08-21 09:06:00| click|
+--------------------+-------+-------------+-------------------+------+-------------------+------+
only showing top 5 rows


In [ ]:
# apenas as combinações em que click_time acontece depois do view_time
filtered = joined.where(F.col("click_time") > F.col("view_time"))
filtered.show(5)

+--------------------+-------+-------------+-------------------+------+-------------------+------+
|             user_id|   adId|   campaignId|          view_time|action|         click_time|action|
+--------------------+-------+-------------+-------------------+------+-------------------+------+
|d9ad512e-274f-41a...|adId_06|campaignId_02|2016-01-01 03:25:00|  view|2016-01-04 08:11:00| click|
|40969830-24a3-4d3...|adId_01|campaignId_02|2016-07-25 09:41:00|  view|2016-12-07 22:18:00| click|
|40969830-24a3-4d3...|adId_01|campaignId_02|2016-02-25 17:05:00|  view|2016-12-07 22:18:00| click|
|40969830-24a3-4d3...|adId_01|campaignId_02|2016-07-25 09:57:00|  view|2016-12-07 22:18:00| click|
|40969830-24a3-4d3...|adId_01|campaignId_02|2016-09-06 20:06:00|  view|2016-12-07 22:18:00| click|
+--------------------+-------+-------------+-------------------+------+-------------------+------+
only showing top 5 rows


In [ ]:
# Cria uma coluna com a diferença de tempo entre view e click (em segundos)
diffed = filtered.withColumn(
    "diff_seconds",
    F.unix_timestamp("click_time") - F.unix_timestamp("view_time")
)
diffed.show(5)

+--------------------+-------+-------------+-------------------+------+-------------------+------+------------+
|             user_id|   adId|   campaignId|          view_time|action|         click_time|action|diff_seconds|
+--------------------+-------+-------------+-------------------+------+-------------------+------+------------+
|d9ad512e-274f-41a...|adId_06|campaignId_02|2016-01-01 03:25:00|  view|2016-01-04 08:11:00| click|      276360|
|40969830-24a3-4d3...|adId_01|campaignId_02|2016-07-25 09:41:00|  view|2016-12-07 22:18:00| click|    11709420|
|40969830-24a3-4d3...|adId_01|campaignId_02|2016-02-25 17:05:00|  view|2016-12-07 22:18:00| click|    24729180|
|40969830-24a3-4d3...|adId_01|campaignId_02|2016-07-25 09:57:00|  view|2016-12-07 22:18:00| click|    11708460|
|40969830-24a3-4d3...|adId_01|campaignId_02|2016-09-06 20:06:00|  view|2016-12-07 22:18:00| click|     7956720|
+--------------------+-------+-------------+-------------------+------+-------------------+------+------

In [ ]:
# Define uma janela particionada por usuário, anúncio, campanha e view_time,
# ordenando pelos menores intervalos de tempo (diff_seconds).
# Isso garante que, para cada view, vamos olhar os clicks subsequentes e escolher o mais próximo.
w = Window.partitionBy("user_id", "adId", "campaignId", "view_time") \
          .orderBy(F.col("diff_seconds").asc())

# atribui um número de linha (row_number) dentro de cada partição.
# O click mais próximo do view recebe rn = 1.
ranked = diffed.withColumn("rn", F.row_number().over(w)) \
               .where(F.col("rn") == 1)
ranked.show(5)

+--------------------+-------+-------------+-------------------+------+-------------------+------+------------+---+
|             user_id|   adId|   campaignId|          view_time|action|         click_time|action|diff_seconds| rn|
+--------------------+-------+-------------+-------------------+------+-------------------+------+------------+---+
|0031aa2d-5988-402...|adId_03|campaignId_03|2016-03-11 20:24:00|  view|2016-03-11 21:12:00| click|        2880|  1|
|0031aa2d-5988-402...|adId_03|campaignId_03|2016-04-05 11:17:00|  view|2016-04-06 09:51:00| click|       81240|  1|
|0031aa2d-5988-402...|adId_03|campaignId_03|2016-04-10 10:31:00|  view|2016-04-10 21:08:00| click|       38220|  1|
|0031aa2d-5988-402...|adId_03|campaignId_03|2016-04-10 19:20:00|  view|2016-04-10 21:08:00| click|        6480|  1|
|0031aa2d-5988-402...|adId_03|campaignId_03|2016-07-14 16:22:00|  view|2016-08-14 20:43:00| click|     2694060|  1|
+--------------------+-------+-------------+-------------------+------+-

In [ ]:
#Agrupa por par (user_id,adId, campaignId) e calculamos a média do tempo entre view e click
avg_times = ranked.groupBy('user_id',"adId", "campaignId") \
                  .agg(F.avg("diff_seconds").alias("avg_diff"))
avg_times.show(5)

+--------------------+-------+-------------+------------------+
|             user_id|   adId|   campaignId|          avg_diff|
+--------------------+-------+-------------+------------------+
|0031aa2d-5988-402...|adId_03|campaignId_03|          635940.0|
|00355f85-a403-4fc...|adId_06|campaignId_03|         3289350.0|
|00437ba9-82bd-41a...|adId_05|campaignId_01|         1414980.0|
|0061c33e-b346-4c6...|adId_03|campaignId_02|3868993.3333333335|
|00640825-4c67-4ec...|adId_09|campaignId_01|2261413.3333333335|
+--------------------+-------+-------------+------------------+
only showing top 5 rows


In [ ]:
#Ordenamos pela menor média e pegamos os 5 primeiros pares.
result = avg_times.orderBy("avg_diff").limit(5)
result.toPandas()

,user_id,adId,campaignId,avg_diff
0,848eb13c-6881-45a3-bd62-05422be20cad,adId_05,campaignId_02,480.0
1,f4db1c5e-7780-4beb-a36f-1322743d99d5,adId_05,campaignId_02,660.0
2,28c5def5-3ede-4ead-ac85-13c1819ead4b,adId_01,campaignId_01,960.0
3,c67d4a61-a911-4982-a685-33664bdcbd76,adId_06,campaignId_02,1140.0
4,ed526924-2c66-470d-a2eb-622eee5e114f,adId_05,campaignId_01,1320.0
